# Amazon Warehouse Analytics — Data Cleaning
**Objective:** Load raw CSVs, inspect quality issues, clean and export analysis-ready data.

**Datasets:** orders, products, inventory, employees, returns

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

DATA_PATH = '../Dataset/'

# Load all datasets
orders    = pd.read_csv(DATA_PATH + 'orders.csv',    parse_dates=['order_date','ship_date','delivery_date','promised_delivery_date'])
products  = pd.read_csv(DATA_PATH + 'products.csv')
inventory = pd.read_csv(DATA_PATH + 'inventory.csv', parse_dates=['last_restock_date'])
employees = pd.read_csv(DATA_PATH + 'employees.csv', parse_dates=['hire_date'])
returns   = pd.read_csv(DATA_PATH + 'returns.csv',   parse_dates=['return_date'])

print('Loaded datasets:')
for name, df in [('orders', orders), ('products', products), ('inventory', inventory),
                  ('employees', employees), ('returns', returns)]:
    print(f'  {name:12s}: {df.shape[0]:,} rows × {df.shape[1]} cols')

## 1. Shape & Data Types

In [ ]:
for name, df in [('orders', orders), ('products', products), ('inventory', inventory),
                  ('employees', employees), ('returns', returns)]:
    print(f'\n{'='*50}')
    print(f'  {name.upper()}')
    print(f'{'='*50}')
    print(df.dtypes)
    print(df.head(3))

## 2. Missing Value Analysis

In [ ]:
def missing_summary(df, name):
    miss = df.isnull().sum()
    miss_pct = (miss / len(df) * 100).round(2)
    result = pd.DataFrame({'Missing': miss, 'Pct': miss_pct})
    result = result[result['Missing'] > 0]
    print(f'\n{name} — {len(result)} columns with missing values:')
    if len(result):
        print(result)
    else:
        print('  No missing values found.')

for name, df in [('orders', orders), ('products', products), ('inventory', inventory),
                  ('employees', employees), ('returns', returns)]:
    missing_summary(df, name)

## 3. Duplicate Detection

In [ ]:
id_cols = {'orders': 'order_id', 'products': 'product_id',
           'inventory': 'inventory_id', 'employees': 'employee_id', 'returns': 'return_id'}

for name, df in [('orders', orders), ('products', products), ('inventory', inventory),
                  ('employees', employees), ('returns', returns)]:
    dupes = df.duplicated().sum()
    pk_dupes = df[id_cols[name]].duplicated().sum()
    print(f'{name:12s} — full duplicates: {dupes}, PK duplicates: {pk_dupes}')

## 4. Outlier Detection

In [ ]:
def iqr_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    return len(outliers), lower, upper

numeric_checks = {
    'orders':    ['revenue', 'quantity', 'processing_time_hours'],
    'products':  ['unit_price', 'unit_cost', 'weight_kg'],
    'employees': ['salary', 'performance_score', 'error_rate']
}

for tbl, (df_name, df) in zip(numeric_checks.keys(),
    [('orders', orders), ('products', products), ('employees', employees)]):
    print(f'\n{tbl.upper()}')
    for col in numeric_checks[tbl]:
        n, lo, hi = iqr_outliers(df, col)
        print(f'  {col:30s}: {n:5,} outliers  (valid range: {lo:.2f} – {hi:.2f})')

## 5. Date Consistency Checks

In [ ]:
# Ship date before order date?
invalid_ship = orders[orders['ship_date'] < orders['order_date']]
print(f'Orders where ship_date < order_date : {len(invalid_ship)}')

# Delivery before ship?
invalid_del = orders[orders['delivery_date'] < orders['ship_date']]
print(f'Orders where delivery_date < ship_date: {len(invalid_del)}')

# Future order dates?
future = orders[orders['order_date'] > pd.Timestamp('today')]
print(f'Orders with future order_date         : {len(future)}')

## 6. Cleaning Steps

In [ ]:
# --- orders ---
# Standardise string columns
for col in ['order_status', 'region', 'sales_channel']:
    orders[col] = orders[col].str.strip().str.title()

# Clip processing_time_hours to realistic range (1–72 hrs)
orders['processing_time_hours'] = orders['processing_time_hours'].clip(lower=1, upper=72)

# Derived columns
orders['order_year']  = orders['order_date'].dt.year
orders['order_month'] = orders['order_date'].dt.month
orders['order_dow']   = orders['order_date'].dt.day_name()
orders['delivery_days_actual']  = (orders['delivery_date'] - orders['order_date']).dt.days
orders['delivery_days_promised']= (orders['promised_delivery_date'] - orders['order_date']).dt.days
orders['delay_days'] = orders['delivery_days_actual'] - orders['delivery_days_promised']

# --- products ---
products['margin_pct'] = ((products['unit_price'] - products['unit_cost']) / products['unit_price'] * 100).round(2)
products['category'] = products['category'].str.strip()

# --- employees ---
employees['tenure_years'] = ((pd.Timestamp('today') - employees['hire_date']).dt.days / 365).round(1)

# --- inventory ---
inventory['utilization_pct'] = (inventory['stock_quantity'] / inventory['max_stock_level'] * 100).round(2)

print('Cleaning complete.')
print(f'  orders   shape: {orders.shape}')
print(f'  products shape: {products.shape}')

## 7. Export Cleaned Data

In [ ]:
import os
out_path = DATA_PATH + 'cleaned/'
os.makedirs(out_path, exist_ok=True)

orders.to_csv(out_path + 'orders_clean.csv', index=False)
products.to_csv(out_path + 'products_clean.csv', index=False)
inventory.to_csv(out_path + 'inventory_clean.csv', index=False)
employees.to_csv(out_path + 'employees_clean.csv', index=False)
returns.to_csv(out_path + 'returns_clean.csv', index=False)

print('Exported cleaned CSVs to Dataset/cleaned/')

## 8. Final Quality Report

In [ ]:
print('=== FINAL DATA QUALITY REPORT ===')
for name, df in [('orders', orders), ('products', products), ('inventory', inventory),
                  ('employees', employees), ('returns', returns)]:
    null_total = df.isnull().sum().sum()
    dupe_total = df.duplicated().sum()
    print(f'{name:12s} | rows: {len(df):>8,} | nulls: {null_total:>5} | dupes: {dupe_total:>5}')